# Python Code For Simulation of Partial Cracking Ammonia Combustion in Counter Flow Diffusion Flame

This notebook simulates the extinction characteristics of a counterflow diffusion flame for partially cracked ammonia. Ammonia (NH3) is a promising carbon-free fuel, and understanding its combustion stability is crucial. "Cracking" refers to the pre-combustion decomposition of ammonia into hydrogen (H2) and nitrogen (N2).

The simulation will vary several parameters to map the flame's stability:

- Ammonia Cracking Fraction (*f*): The proportion of ammonia that has been cracked into hydrogen and nitrogen.
- Fuel and Air Inlet Velocities *(u_fuel, u_air)*: The speed at which the fuel and air streams enter the counterflow burner.
- Burner Width: The distance between the fuel and air inlets.

The goals are:

- Run a series of flame simulations for each parameter combination using the Cantera library.
- Determine if a flame is stable or extinct based on its maximum temperature.
- Store the results, including the maximum temperature of each case, in a structured Excel file suitable for machine learning analysis.
- Generate and save 2D "extinction maps" that visualize the stability boundaries.
- This process is computationally intensive, so the script leverages the multiprocessing module to run simulations in parallel, significantly reducing the total time.

## Import Libraries
This cell imports all the necessary Python libraries for the simulation, data handling, and plotting

In [1]:
import itertools as it
import multiprocessing as mp
from pathlib import Path
import time

# Cantera is the core library for chemical kinetics and flame simulation
import cantera as ct

# NumPy and Pandas are used for numerical operations and data management
import numpy as np
import pandas as pd

# Matplotlib is used for plotting the results
import matplotlib.pyplot as plt

## User Settings and Global Constants
Here, you can configure all the key parameters for the simulation.

In [2]:
# --- SIMULATION SETTINGS ---
MECH           = "mechanisms/reactionsokafar.yaml"  # Chemical mechanism file (must be in the same directory or provide full path)
P_ATM          = ct.one_atm              # Pressure [Pa]
T_FUEL         = 300.0                   # Fuel inlet temperature [K]
T_AIR          = 300.0                   # Air inlet temperature [K]
TEMP_THRESHOLD = 1273.0                  # Extinction threshold temperature [K]

# --- PARALLEL PROCESSING ---
# Use one less than the total number of CPU cores, or at least 1
CPUS = max(mp.cpu_count() - 1, 1)

# --- PARAMETER RANGES TO SIMULATE ---
# Cracking fractions (f=0 is pure NH3, f=1 is fully cracked)
crack_fractions = [0.28, 0.50, 0.75]
# Range of fuel velocities [m/s]
uf_range        = np.arange(0.0, 30.0 + 1e-9, 1.0)
# Range of air velocities [m/s]
ua_range        = np.arange(0.0, 18.0 + 1e-9, 1.0)
# Burner widths [meters]
widths          = [0.01, 0.02, 0.03]      # Corresponds to 1 cm, 2 cm, 3 cm

# --- OUTPUT CONFIGURATION ---
OUT_XLSX = Path("data/extinction_data.xlsx")
FIG_DIR  = Path("results/figures")
# Create the 'results/figures' directory if it doesn't already exist
FIG_DIR.mkdir(exist_ok=True)

print("Settings configured successfully.")
print(f"Using chemical mechanism: {MECH}")
print(f"Simulations will run on {CPUS} CPU core(s).")

Settings configured successfully.
Using chemical mechanism: mechanisms/reactionsokafar.yaml
Simulations will run on 11 CPU core(s).


## Composition and Helper Functions
These functions help define the chemical compositions and handle edge cases in the simulation setup.

In [3]:
def cracked_fuel_X(f):
    """
    Returns the mole fraction string for partially cracked ammonia fuel.
    The reaction is: NH3 -> (1-f)NH3 + 1.5*f*H2 + 0.5*f*N2
    """
    return f"NH3:{1 - f}, H2:{1.5 * f}, N2:{0.5 * f}"

# Define standard air composition
AIR_X = "O2:0.21, N2:0.79"

def fix_velocity(u):
    """
    Cantera's mass flow controllers can't have a zero velocity.
    This function replaces 0 with a very small number (1e-9) for the simulation.
    """
    return max(u, 1e-9)

print("Helper functions defined.")

Helper functions defined.


## The Core Simulation Function *(run_case)*
This is the heart of the simulation. The *run_case* function takes a set of parameters (a single case), sets up a Cantera *CounterflowDiffusionFlame*, solves it, and returns the key results. A *try...except* block gracefully handles cases where the flame solver fails to converge.

In [4]:
def run_case(args):
    """
    Runs a single counterflow flame simulation for a given case.
    
    Args:
        A tuple containing (f, uf, ua, width).
    
    Returns:
        A tuple with results: (f, uf, ua, width, Tmax, stable).
    """
    f, uf, ua, width = args
    
    # Initialize a Cantera Solution object with the specified mechanism
    gas = ct.Solution(MECH)

    # Calculate fuel-side mass flow rate
    gas.TPX = T_FUEL, P_ATM, cracked_fuel_X(f)
    rho_f = gas.density
    mdot_f = rho_f * uf

    # Calculate air-side mass flow rate
    gas.TPX = T_AIR, P_ATM, AIR_X
    rho_a = gas.density
    mdot_a = rho_a * ua

    # Create the flame object
    flame = ct.CounterflowDiffusionFlame(gas, width=width)
    flame.P = P_ATM

    # Set fuel inlet properties
    flame.fuel_inlet.mdot = mdot_f
    flame.fuel_inlet.T = T_FUEL
    flame.fuel_inlet.X = cracked_fuel_X(f)

    # Set oxidizer (air) inlet properties
    flame.oxidizer_inlet.mdot = mdot_a
    flame.oxidizer_inlet.T = T_AIR
    flame.oxidizer_inlet.X = AIR_X

    # Set an initial guess for the solution profile
    flame.set_initial_guess()

    # Try to solve the flame. If it fails, mark as extinct.
    try:
        # loglevel=0 suppresses verbose solver output
        # auto=True helps the solver find a solution automatically
        flame.solve(loglevel=0, auto=True)
        Tmax = float(np.max(flame.T))
        # Check if the maximum temperature is above the stability threshold
        stable = int(Tmax >= TEMP_THRESHOLD)
    except Exception:
        # If any Cantera error occurs, the flame is considered non-convergent/extinct
        Tmax = np.nan
        stable = 0

    return f, uf, ua, width, Tmax, stable

print("Core simulation function `run_case` is ready.")

Core simulation function `run_case` is ready.


## Run the Full Simulation Batch
This cell orchestrates the entire simulation batch. It generates all possible combinations of parameters, distributes them to a pool of CPU workers for parallel execution, and gathers the results into a pandas DataFrame.

In [ ]:
def print_progress(count, total, f, uf, ua, w, Tmax, elapsed):
    """Formats and prints a dynamic progress line."""
    # Build the progress string
    progress_percent = (count / total) * 100
    # Note the spaces at the end of the string to clear previous longer lines
    status_line = (
        f"\rProgress: {count}/{total} ({progress_percent:.1f}%) | "
        f"Last Case: [f={f:.2f}, uf={uf:.1f}, ua={ua:.1f}, w={w*100:.0f}cm] -> T_max={Tmax:.0f}K | "
        f"Elapsed: {elapsed/60:.1f}min  "
    )
    # Print to the same line without a newline character
    print(status_line, end="", flush=True)

start_time = time.perf_counter()

# Generate all unique combinations of parameters
all_params = list(it.product(crack_fractions, uf_range, ua_range, widths))
total_cases = len(all_params)
print(f"Setting up to run {total_cases} total simulation cases...")

# Prepare the list of tasks with corrected velocities
tasks = [(f, fix_velocity(uf), fix_velocity(ua), w) for f, uf, ua, w in all_params]
all_results = []

# Use a multiprocessing Pool to run cases in parallel
print(f"Running simulations on {CPUS} CPU core(s). This may take some time...")
with mp.Pool(CPUS) as pool:
    # imap_unordered is efficient for collecting results as they complete
    for i, res in enumerate(pool.imap_unordered(run_case, tasks), 1):
        all_results.append(res)
        
        # Unpack the result to get case details
        f_res, uf_res, ua_res, w_res, Tmax_res, _ = res
        
        # Calculate elapsed time for the progress report
        current_elapsed = time.perf_counter() - start_time
        
        # Call the dynamic progress printer for EVERY case
        print_progress(i, total_cases, f_res, uf_res, ua_res, w_res, Tmax_res, current_elapsed)
        
        # Optional: Print a permanent log entry every 100 cases
        if i % 100 == 0:
            # Print a newline to move off the dynamic status line
            print() 
            # Print the log entry
            print(f"  Milestone: Completed {i}/{total_cases} cases.")
            

# Print a final newline to clear the progress bar
print("\n\nAll simulations completed.")

# Create the final DataFrame
df = pd.DataFrame(all_results,
                  columns=["f_crack", "u_fuel", "u_air", "width", "T_max", "stable"])
# Sort the DataFrame for organized viewing and saving
df.sort_values(["f_crack", "width", "u_fuel", "u_air"], inplace=True)

# TIME REPORT
end_time = time.perf_counter()
elapsed = end_time - start_time
print(f"🕒 Total simulation time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


# Display the first few rows of the resulting DataFrame
print("\nSample of the results data:")
df.head()

Setting up to run 5301 total simulation cases...
Running simulations on 11 CPU core(s). This may take some time...


## Save Results to Excel File
The resulting DataFrame is now saved to an Excel file. The file will contain one sheet named *"all_data"* with the complete dataset, which is perfect for training a machine learning model. It will also contain separate sheets for each *(f_crack, width)* combination for easier manual inspection.

In [ ]:
print(f"Saving data to Excel file: {OUT_XLSX}")

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xl:
    # Save the complete dataset to the "all_data" sheet
    df.to_excel(xl, sheet_name="all_data", index=False)
    
    # Save a separate sheet for each cracking fraction and width combination
    for f_val in crack_fractions:
        for width_val in widths:
            sheet_name = f"f_{f_val:.2f}_w_{int(width_val*100)}"
            subset_df = df[(df.f_crack == f_val) & (df.width == width_val)]
            subset_df.to_excel(xl, sheet_name=sheet_name, index=False)

print(f"\n✅ Results successfully saved ➜ {OUT_XLSX.resolve()}")

## Generate and Save Extinction Maps
This final cell iterates through the results for each *(f_crack, width)* pair and generates a 2D plot. These plots visually represent the flame stability limits, with fuel velocity on the x-axis and air velocity on the y-axis. Each plot is displayed in the notebook and saved as a PNG image in the results/figures directory.

In [ ]:
print("Generating and saving plots...")

for f_val in crack_fractions:
    for width_val in widths:
        # Get the data for the current case
        d = df[(df.f_crack == f_val) & (df.width == width_val)].copy()

        # Replace small simulation velocities (1e-9) back to 0 for clear plotting
        d["u_fuel_plot"] = d["u_fuel"].apply(lambda x: 0 if x <= 1e-9 else x)
        d["u_air_plot"]  = d["u_air"].apply(lambda x: 0 if x <= 1e-9 else x)

        plt.figure(figsize=(7, 6))

        # --- Plotting Section ---
        # Stable points (blue circles)
        stable_data = d[d.stable == 1]
        plt.scatter(stable_data.u_fuel_plot, stable_data.u_air_plot,
                    marker="o", label="Stable flame", s=30, c='blue', zorder=2)

        # Extinct points (red crosses)
        extinct_data = d[d.stable == 0]
        plt.scatter(extinct_data.u_fuel_plot, extinct_data.u_air_plot,
                    marker="x", label="Extinct / no-solution", s=30, c='red', zorder=2)
        
        # Boundary line (green line with markers)
        if not stable_data.empty:
            boundary_df = stable_data.groupby("u_fuel_plot")["u_air_plot"].max().reset_index()
            boundary_df.sort_values(by="u_fuel_plot", inplace=True)
            plt.plot(boundary_df.u_fuel_plot, boundary_df.u_air_plot,
                     linestyle='-', color='green', marker='.', linewidth=2,
                     label="Stability Limit", zorder=3)
        
        # --- Formatting ---
        plt.grid(True, ls=":", zorder=1)
        plt.xlabel("Fuel velocity $u_{fuel}$  [m/s]")
        plt.ylabel("Air velocity $u_{air}$  [m/s]")
        plt.title(f"Extinction Map\nf = {f_val:.2f}, width = {width_val*100:.0f} mm")
        plt.legend()
        plt.tight_layout()
        
        # Save figure and show it
        fname = FIG_DIR / f"extinction_map_f_{f_val:.2f}_w_{int(width_val*100)}.png"
        plt.savefig(fname, dpi=300)
        plt.show() # Display the plot in the notebook
        plt.close() # Close the plot to free memory
        print(f"  - Figure saved ➜ {fname}")
        
print("\nAll plots have been generated and saved.")